# SPY 바이앤홀드 외부 교차 확인

스펙 '외부 교차 확인': SPY 결과를 공개된 SPY 총수익률(배당 재투자)과 비교해 연 0.1%p 이내인지 확인.

**공개 값 출처**: State Street(SSGA) SPY 펀드 페이지의 성과 데이터 (`https://www.ssga.com/bin/v1/ssmp/fund/fundfinder?country=us&language=en&role=intermediary&product=etfs&ui=fund-finder`, 2026-11-08 조회). 시장가격(Market Value) 기준 연환산 총수익률.

비교 두 가지:
- **종가→종가**: 수정주가 데이터가 맞는지 (공개 값과 같은 정의)
- **엔진 바이앤홀드**: 비용 0bp. 엔진은 시작일 종가 다음 날 **시가**에 사므로, 시작일 종가→다음 날 시가 사이 움직임만큼 공개 값과 달라진다. 이 차이를 따로 계산해 엔진 오차가 아닌지 확인한다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from swinglab.backtest import run_backtest
from swinglab.config import load_backtest_config, load_universe
from swinglab.pipeline.storage import read_prices
from swinglab.strategies import BuyAndHold, build_strategy

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
universe = load_universe(ROOT / "config")
config = load_backtest_config(ROOT / "config")
prices = read_prices(ROOT / "data/clean/prices.parquet")  # 엔진 입력
print(f"데이터: {prices['date'].min().date()} ~ {prices['date'].max().date()}, {prices['ticker'].nunique()}종목")

데이터: 2010-01-04 ~ 2026-11-06, 20종목


In [2]:
# SSGA 공개 값: SPY Market Value 연환산 총수익률(%)
PUBLISHED = {
    "2026-09-30": {1: 15.66, 3: 22.73, 5: 13.66, 10: 15.18},
    "2026-10-31": {1: 11.97, 3: 23.25, 5: 11.93, 10: 15.29},
}
spy = prices[prices["ticker"] == "SPY"].set_index("date")
nocost = config.model_copy(update={"cost_bps": 0.0})

rows = []
for asof, perf in PUBLISHED.items():
    end = spy.index[spy.index <= pd.Timestamp(asof)][-1]
    for years, published in perf.items():
        start = spy.index[spy.index <= pd.Timestamp(asof) - pd.DateOffset(years=years)][-1]
        first_trade = spy.index[spy.index > start][0]
        c2c = (spy.at[end, "close"] / spy.at[start, "close"]) ** (1 / years) - 1

        window = nocost.model_copy(update={"start": first_trade.date(), "end": end.date()})
        r = run_backtest(prices, BuyAndHold("SPY"), window)
        engine = (r.equity.iloc[-1] / r.equity.iloc[0]) ** (1 / years) - 1
        # 엔진은 다음 날 시가에 산다. 시가/시작일 종가(갭)를 곱해 되돌리면 종가→종가와 같아야 한다
        gap = spy.at[first_trade, "open"] / spy.at[start, "close"]
        engine_gap_adj = ((r.equity.iloc[-1] / r.equity.iloc[0]) * gap) ** (1 / years) - 1
        rows.append({
            "기준일": asof, "기간(년)": years, "시작 종가일": start.date(), "끝": end.date(),
            "공개(%)": published,
            "종가→종가(%)": round(c2c * 100, 3), "차이(%p)": round(c2c * 100 - published, 3),
            "엔진 B&H(%)": round(engine * 100, 3), "엔진 차이(%p)": round(engine * 100 - published, 3),
            "첫날 갭(%)": round((gap - 1) * 100, 3),
            "엔진 갭 보정(%)": round(engine_gap_adj * 100, 3),
        })
table = pd.DataFrame(rows)
table

,기준일,기간(년),시작 종가일,끝,공개(%),종가→종가(%),차이(%p),엔진 B&H(%),엔진 차이(%p),첫날 갭(%),엔진 갭 보정(%)
0,2026-09-30,1,2025-09-30,2026-09-30,15.66,15.713,0.053,16.238,0.578,-0.452,15.713
1,2026-09-30,3,2023-09-29,2026-09-30,22.73,22.757,0.027,22.840,0.110,-0.201,22.757
2,2026-09-30,5,2021-09-30,2026-09-30,13.66,13.698,0.038,13.601,-0.059,0.429,13.698
3,2026-09-30,10,2016-09-30,2026-09-30,15.18,15.234,0.054,15.259,0.079,-0.222,15.234
4,2026-10-31,1,2025-10-31,2026-10-30,11.97,11.995,0.025,11.405,-0.565,0.529,11.995
5,2026-10-31,3,2023-10-31,2026-10-30,23.25,23.284,0.034,23.186,-0.064,0.239,23.284
6,2026-10-31,5,2021-10-29,2026-10-30,11.93,11.962,0.032,11.911,-0.019,0.229,11.962
7,2026-10-31,10,2016-10-31,2026-10-30,15.29,15.331,0.041,15.310,0.020,0.179,15.331


In [3]:
tol = 0.1
data_ok = (table["차이(%p)"].abs() <= tol).all()
gap_explains = np.allclose(table["엔진 갭 보정(%)"], table["종가→종가(%)"], atol=1e-3)
ten_year_ok = (table.loc[table["기간(년)"] == 10, "엔진 차이(%p)"].abs() <= tol).all()
print(f"수정주가 종가→종가 vs 공개 값: 8개 구간 모두 ±{tol}%p 이내 → {data_ok} (최대 {table['차이(%p)'].abs().max():.3f}%p)")
print(f"엔진 B&H 10년 구간 ±{tol}%p 이내 → {ten_year_ok}")
print(f"엔진과 종가→종가의 차이가 첫날 갭만으로 설명됨 → {gap_explains}")

수정주가 종가→종가 vs 공개 값: 8개 구간 모두 ±0.1%p 이내 → True (최대 0.054%p)
엔진 B&H 10년 구간 ±0.1%p 이내 → True
엔진과 종가→종가의 차이가 첫날 갭만으로 설명됨 → True


## 해석

- 수정주가 데이터는 공개 총수익률과 모든 구간에서 0.1%p 이내로 맞는다.
- 엔진 바이앤홀드는 10년 구간에서 0.1%p 이내다. 1·3년처럼 짧은 구간은 첫 체결이 다음 날 시가라서 첫날 갭이 연환산 값에 크게 실리며, 이 차이를 되돌리면 종가→종가와 일치한다 (엔진 계산 오차가 아님).
- 공개 값은 월말 기준이라 백테스트 전체 구간(2010-01-05~)과 같은 구간의 공개 값은 없다.